# KURE 번아웃 End-to-End 파인튜닝 v1
**목적**: KURE 백본을 번아웃 분류 태스크에 맞게 직접 파인튜닝하여 Stage 2 F1 개선

## 기존 방식 vs E2E
| | 기존 (frozen) | E2E (이 노트북) |
|--|--|--|
| KURE | 고정 (범용 임베딩) | 상위 2개 레이어 학습 |
| 임베딩 캐시 | 가능 | 불가 (매 에폭 forward) |
| 학습 대상 | MLP 헤드만 | 백본 일부 + MLP 헤드 |
| 메모리 | 적음 | 중간 (gradient checkpointing 적용) |

## 메모리 절약 전략 (Colab Free 대응)
- KURE 상위 **2개 레이어만** 해제 (전체 파인튜닝 대비 메모리 절약)
- **Mixed precision (fp16)** 학습
- **Gradient checkpointing** 활성화
- Batch size = 16

## 파이프라인
```
stage2_train_v3.csv
        ↓ 토크나이징
KURE 백본 (상위 2레이어 학습) → mean pooling → MLP 분류기
        ↓ warm-start from stage2_model_v3.pt
Stage 2 F1 0.4754 기준 개선 목표
```

## 1. 환경 설정

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate sentence-transformers scikit-learn

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
warnings.filterwarnings('ignore')

# 메모리 단편화 방지
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Burnout'

STAGE2_CATEGORIES = {0: '정서적_고갈', 1: '좌절_압박', 2: '부정적_대인관계', 3: '자기비하'}

S2_TRAIN_PATH  = f'{DATA_PATH}/stage2_train_v3.csv'
S2_VAL_PATH    = f'{DATA_PATH}/stage2_val_v3.csv'
S2_MODEL_PATH  = f'{DATA_PATH}/stage2_model_v3.pt'     # 분류기 warm-start용
SAVE_MODEL_PATH = f'{DATA_PATH}/stage2_model_e2e_v1.pt' # E2E 학습 결과

print('경로 확인:')
for name, path in [('S2 Train', S2_TRAIN_PATH), ('S2 Val', S2_VAL_PATH),
                   ('S2 Model (warm-start)', S2_MODEL_PATH)]:
    print(f'  {"✅" if os.path.exists(path) else "❌"} {name}: {path}')

## 3. 데이터 로드 + 토크나이징

In [ ]:
s2_train = pd.read_csv(S2_TRAIN_PATH)
s2_val   = pd.read_csv(S2_VAL_PATH)

print(f'Stage 2 Train: {len(s2_train):,}건')
print(f'Stage 2 Val  : {len(s2_val):,}건')
print()
print('Train 클래스 분포:')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train['label'] == label).sum()
    pct = cnt / len(s2_train) * 100
    print(f'  {cat}: {cnt:,}건 ({pct:.1f}%)')

In [ ]:
# KURE 토크나이저 로드
print('KURE 로딩 중...')
st_model = SentenceTransformer('nlpai-lab/KURE-v1')
tokenizer = st_model.tokenizer
print('✅ 토크나이저 로드 완료')

MAX_LEN = 128

def tokenize(texts):
    return tokenizer(
        texts, padding='max_length', truncation=True,
        max_length=MAX_LEN, return_tensors='pt'
    )

print('Train 토크나이징 중...')
train_enc = tokenize(s2_train['text'].tolist())
print('Val 토크나이징 중...')
val_enc   = tokenize(s2_val['text'].tolist())

train_labels = torch.tensor(s2_train['label'].values, dtype=torch.long)
val_labels   = torch.tensor(s2_val['label'].values,   dtype=torch.long)

train_ds = TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], train_labels)
val_ds   = TensorDataset(val_enc['input_ids'],   val_enc['attention_mask'],   val_labels)

print(f'\n토크나이징 완료')
print(f'  Train: {train_enc["input_ids"].shape}')
print(f'  Val  : {val_enc["input_ids"].shape}')

## 4. 모델 정의 (E2E)

- KURE 백본: 상위 2개 레이어만 학습 가능, 나머지 고정
- 분류기 헤드: stage2_model_v3.pt에서 warm-start
- Gradient checkpointing 활성화 (메모리 절약)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class E2EBurnoutModel(nn.Module):
    def __init__(self, backbone, hidden_dim=256, num_classes=4, dropout=0.2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(1024, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def mean_pool(self, token_embeds, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        return self.classifier(pooled)

In [ ]:
NUM_UNFREEZE = 2  # 상위 N개 레이어 학습

# SentenceTransformer에서 백본 추출
backbone = st_model[0].auto_model
del st_model  # 메모리 절약
torch.cuda.empty_cache()

# 전체 고정
for param in backbone.parameters():
    param.requires_grad = False

# 상위 N개 레이어 해제
layers = None
if hasattr(backbone, 'encoder') and hasattr(backbone.encoder, 'layer'):
    layers = backbone.encoder.layer
elif hasattr(backbone, 'roberta'):
    layers = backbone.roberta.encoder.layer

if layers is not None:
    for layer in layers[-NUM_UNFREEZE:]:
        for param in layer.parameters():
            param.requires_grad = True
    print(f'✅ 상위 {NUM_UNFREEZE}개 레이어 학습 가능 (전체 {len(layers)}개 중)')
else:
    print('⚠️ 레이어 구조 감지 실패 → 백본 전체 고정')

# Gradient checkpointing (메모리 절약)
if hasattr(backbone, 'gradient_checkpointing_enable'):
    backbone.gradient_checkpointing_enable()
    print('✅ Gradient checkpointing 활성화')

# E2E 모델 생성
model = E2EBurnoutModel(backbone, hidden_dim=256, num_classes=4, dropout=0.2).to(device)

# 분류기 warm-start
if os.path.exists(S2_MODEL_PATH):
    ckpt = torch.load(S2_MODEL_PATH, map_location='cpu', weights_only=False)
    raw = ckpt['model_state_dict']
    # 'classifier.' prefix 제거
    cls_state = {k.replace('classifier.', ''): v for k, v in raw.items()}
    model.classifier.load_state_dict(cls_state)
    print(f'✅ 분류기 warm-start: {S2_MODEL_PATH}')
else:
    print('⚠️ warm-start 모델 없음 → random init')

# 파라미터 수 요약
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n파라미터: 전체 {total/1e6:.1f}M | 학습 가능 {trainable/1e6:.1f}M ({trainable/total*100:.1f}%)')
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 5. 학습

In [ ]:
E2E_CONFIG = {
    'epochs': 30,
    'batch_size': 16,
    'lr_backbone': 1e-5,   # 백본: 매우 작게
    'lr_head': 1e-4,       # 헤드: 상대적으로 크게
    'weight_decay': 1e-4,
    'patience': 5,
    'warmup_epochs': 2,
    'focal_gamma': 2.0,
    'label_smoothing': 0.05,
}

train_loader = DataLoader(train_ds, batch_size=E2E_CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

counts = torch.bincount(train_labels).float()
class_weights = (1.0 / counts).to(device)
class_weights = class_weights / class_weights.sum() * 4
criterion = FocalLoss(gamma=E2E_CONFIG['focal_gamma'], weight=class_weights,
                      label_smoothing=E2E_CONFIG['label_smoothing'])

backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_params     = list(model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': E2E_CONFIG['lr_backbone'],
     'weight_decay': E2E_CONFIG['weight_decay']},
    {'params': head_params,     'lr': E2E_CONFIG['lr_head'],
     'weight_decay': E2E_CONFIG['weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=E2E_CONFIG['epochs']
)
scaler = GradScaler()  # fp16 mixed precision

print(f'클래스 가중치: {class_weights.cpu().numpy().round(3)}')
print(f'Train 배치 수: {len(train_loader)}')

In [ ]:
best = {'f1': 0.0, 'acc': 0.0, 'epoch': 0}
patience_cnt = 0

print('학습 시작')
print('=' * 65)

for epoch in range(E2E_CONFIG['epochs']):
    # Warmup LR
    if epoch < E2E_CONFIG['warmup_epochs']:
        scale = (epoch + 1) / E2E_CONFIG['warmup_epochs']
        optimizer.param_groups[0]['lr'] = E2E_CONFIG['lr_backbone'] * scale
        optimizer.param_groups[1]['lr'] = E2E_CONFIG['lr_head'] * scale

    # Train
    model.train()
    train_loss = 0.0
    for input_ids, attention_mask, lbl in train_loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        lbl            = lbl.to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    if epoch >= E2E_CONFIG['warmup_epochs']:
        scheduler.step()

    # Validate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, lbl in val_loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            with autocast():
                logits = model(input_ids, attention_mask)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(lbl.tolist())

    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    val_f1  = f1_score(all_labels, all_preds, average='macro')

    improved = val_f1 > best['f1']
    if improved:
        best = {'f1': val_f1, 'acc': val_acc, 'epoch': epoch + 1}
        torch.save({
            'model_state_dict': model.state_dict(),
            'num_unfreeze_layers': NUM_UNFREEZE,
            'num_classes': 4,
            'categories': STAGE2_CATEGORIES,
            'config': E2E_CONFIG,
            'best_metrics': best,
            'data_version': 'e2e_v1',
        }, SAVE_MODEL_PATH)
        patience_cnt = 0
        marker = ' ★ BEST'
    else:
        patience_cnt += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or improved:
        print(f'Epoch {epoch+1:3d} | Loss {train_loss:.4f} | '
              f'Acc {val_acc:.4f} | F1-macro {val_f1:.4f}{marker}')

    if patience_cnt >= E2E_CONFIG['patience']:
        print(f'\nEarly stopping at epoch {epoch + 1}')
        break

print('=' * 65)
print(f'최고: Epoch {best["epoch"]} | F1 {best["f1"]:.4f} | Acc {best["acc"]:.4f}')

## 6. 성능 비교

In [ ]:
ckpt = torch.load(SAVE_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for input_ids, attention_mask, lbl in val_loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        with autocast():
            logits = model(input_ids, attention_mask)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(lbl.tolist())

print('─── Classification Report ───')
print(classification_report(
    all_labels, all_preds,
    target_names=list(STAGE2_CATEGORIES.values()), digits=4
))

final_f1    = f1_score(all_labels, all_preds, average='macro')
baseline_f1 = 0.4754
improvement = final_f1 - baseline_f1

print('=' * 60)
print('📊 최종 성능 비교')
print(f'  Stage 2 v3 (기준)        : F1 = {baseline_f1:.4f}')
print(f'  Stage 2 FineTune v4      : F1 = 0.4839')
print(f'  Stage 2 StyleTransfer v2 : F1 = 0.4690')
print(f'  Stage 2 E2E v1           : F1 = {final_f1:.4f}')
print(f'  v3 대비 개선              : {improvement:+.4f}')
print()
if improvement > 0.02:
    print('  ✅ E2E 효과 있음 → stage2_model_e2e_v1.pt 사용 권장')
elif improvement > 0:
    print('  ↔ 미미한 향상 → 추가 실험 고려')
else:
    print('  ❌ 개선 없음 → unfreeze 레이어 수 조정 또는 다른 접근 필요')
print('=' * 60)

## 7. 직접 문장 테스트

In [ ]:
TEST_SENTENCES = [
    ('잠을 못 잤더니 지쳤다',               '정서적_고갈'),
    ('오늘도 야근. 몸이 한계다',             '정서적_고갈'),
    ('팀장이 또 뭐라 했다. 짜증',            '좌절_압박'),
    ('왜 나만 이렇게 힘들까. 내 탓인가',     '자기비하'),
    ('동료가 내 공을 가로챘다. 억울',        '부정적_대인관계'),
    ('아무것도 하기 싫다. 그냥 누워있고 싶어', '정서적_고갈'),
]

model.eval()
print(f'[E2E v1 모델 — {SAVE_MODEL_PATH}]\n')
print(f'  {"문장":<32} {"예측":<14} {"정답":<14} {"신뢰도":<8} OK?')
print('-' * 75)

with torch.no_grad():
    for text, gt in TEST_SENTENCES:
        enc = tokenizer(
            [text], padding='max_length', truncation=True,
            max_length=MAX_LEN, return_tensors='pt'
        )
        input_ids      = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)
        with autocast():
            logits = model(input_ids, attention_mask)
        probs    = torch.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
        pred_cat = STAGE2_CATEGORIES[pred_idx]
        conf     = probs[pred_idx].item()
        ok       = '✅' if pred_cat == gt else '❌'
        print(f'  {text:<32} {pred_cat:<14} {gt:<14} {conf:.1%:<8} {ok}')